# Persuasion Index Analysis Example

This notebook shows the standard ways to run the Persuasion Index scorer for a single argument, a DataFrame of arguments, seeded lexicons, and the UKP logistic-regression weighting profile.


## 1. Setup

Run this cell first. The convenience API uses the expanded lexicons by default. Use `lexicon="seeded"` or `use_expanded_lexicons=False` only when you want the original seed lexicons.


In [ ]:
import pandas as pd

from persuasion_runner import build_score_matrices, score_persuasion
from persuasion_profile import get_persuasion_report


## 2. Score One Argument

For a single string, `score_persuasion` returns the raw nested category dictionary by default. Each category includes sub-feature scores and a `mean`.


In [ ]:
text = "Sample text to analyze for persuasion features."

# Default behavior: expanded lexicons + raw nested score dictionary.
single_scores = score_persuasion(text)
single_scores


If you want DataFrame output for one text, set `output="matrices"`. This returns the same two matrices used for batch analysis.


In [ ]:
# One-row matrices for a single argument.
single_sub, single_mean = score_persuasion(text, output="matrices")
single_mean


## 3. Score a DataFrame

For a DataFrame, provide the text column name. The function returns two DataFrames: one with all sub-features and one with category means.


In [ ]:
example_df = pd.DataFrame({
    "argument": [
        "I think this product is amazing and will change your life!",
        "This is the worst experience I've ever had. Do not buy this.",
        "It's okay, not great but not terrible either.",
        "I have no strong feelings about this one way or the other.",
    ]
})

example_df


In [ ]:
# DataFrame input defaults to expanded lexicons and returns matrices.
X_sub, X_mean = score_persuasion(example_df, text_col="argument")


In [ ]:
# Every individual persuasion sub-feature, such as Evidence.statistical.
X_sub


In [ ]:
# Category-level means, such as Evidence.mean.
X_mean


## 4. Use Seeded Lexicons Instead

The expanded lexicons are recommended as the default. Use seeded/original lexicons when you need to reproduce seed-only results or compare against the expanded version.


In [ ]:
# Option A: through the main convenience function.
seeded_sub, seeded_mean = score_persuasion(
    example_df,
    text_col="argument",
    lexicon="seeded",
)

seeded_mean


In [ ]:
# Option B: through the matrix-specific helper.
seeded_sub_2, seeded_mean_2 = build_score_matrices(
    example_df,
    text_col="argument",
    use_expanded_lexicons=False,
)


## 5. UKP Logistic-Regression Persuasion Profile

`get_persuasion_report` applies stored UKP logistic-regression weights to the raw scores. It returns `(raw_scores, ukp_weighted)`. The weighted result includes two probability estimates: one from sub-feature weights and one from category-mean weights.


In [ ]:
profile_text = example_df.loc[0, "argument"]

# Defaults to expanded lexicons. Set use_expanded_lexicons=False for seeded lexicons.
raw_scores, ukp_weighted = get_persuasion_report(
    profile_text,
    use_expanded_lexicons=True,
)

if ukp_weighted is None:
    raise FileNotFoundError(
        "UKP coefficient files were not found in helper_features/regression_outputs."
    )

ukp_weighted["metadata"]


In [ ]:
# Present the final logistic-regression outputs in a compact table.
ukp_summary = pd.DataFrame(ukp_weighted["metadata"]).T
ukp_summary


In [ ]:
# Optional: inspect the weighted contribution from one category.
ukp_weighted["Evidence"]
